# **CB 2 Metode Numerik**

**Skenario:** Jaringan 6 router dengan hukum konservasi aliran paket (aliran masuk = aliran keluar).

Variabel keputusan $x_1, x_2, x_3, x_4, x_5, x_6$ menyatakan total laju aliran paket di masing-masing router (paket/detik).

In [ ]:
# setup
import numpy as np
import time
from copy import deepcopy

### **1. Definisi Matriks A dan Vektor b**

Berdasarkan persamaan aliran dari soal 1 & 2 (setelah dipindahkan ke bentuk baku $Ax = b$):

$$
\begin{align*}
x_1 - 0.2x_2 - 0.3x_3 &= 50 \\
-0.4x_1 + x_2 - 0.1x_4 &= 20 \\
-0.2x_1 + x_3 - 0.4x_5 &= 30 \\
-0.3x_2 + x_4 - 0.2x_6 &= 10 \\
-0.3x_3 + x_5 - 0.3x_6 &= 40 \\
-0.4x_4 - 0.2x_5 + x_6 &= 60
\end{align*}
$$

Sehingga matriks koefisien $A$ dan vektor konstanta $b$ adalah:

In [ ]:
# Matriks koefisien A (6x6)
A = np.array([
    [ 1.0, -0.2, -0.3,  0.0,  0.0,  0.0],
    [-0.4,  1.0,  0.0, -0.1,  0.0,  0.0],
    [-0.2,  0.0,  1.0,  0.0, -0.4,  0.0],
    [ 0.0, -0.3,  0.0,  1.0,  0.0, -0.2],
    [ 0.0,  0.0, -0.3,  0.0,  1.0, -0.3],
    [ 0.0,  0.0,  0.0, -0.4, -0.2,  1.0]
], dtype=float)

# Vektor konstanta b (bentuk kolom)
b = np.array([50, 20, 30, 10, 40, 60], dtype=float).reshape(-1, 1)

print("Matriks A:")
print(A)
print("\nVektor b:")
print(b)

### **2. Pemeriksaan Syarat Konvergensi Gauss-Seidel (Diagonal Dominance)**

Syarat cukup agar iterasi Gauss-Seidel konvergen adalah matriks $A$ **diagonally dominant**, yaitu:

$$
|a_{ii}| \geq \sum_{j \neq i} |a_{ij}| \quad \text{untuk setiap baris } i
$$

Kita cek satu per satu:

In [ ]:
def cek_diagonal_dominance(A):
    """Memeriksa apakah matriks A diagonally dominant."""
    n = A.shape[0]
    dominant = True
    print("Pemeriksaan Diagonal Dominance:")
    print("-" * 50)
    for i in range(n):
        diag = abs(A[i, i])
        off_diag = np.sum(np.abs(A[i, :])) - diag
        status = "OK" if diag >= off_diag else "TIDAK DOMINAN"
        print(f"Baris {i+1}: |a_ii| = {diag:.1f}  ≥  sum|a_ij| = {off_diag:.1f}  →  {status}")
        if diag < off_diag:
            dominant = False
    print("-" * 50)
    if dominant:
        print("Kesimpulan: Matriks A MEMENUHI syarat diagonally dominant.")
        print("Iterasi Gauss-Seidel dijamin konvergen.")
    else:
        print("Kesimpulan: Matriks A TIDAK diagonally dominant.")
    return dominant

cek_diagonal_dominance(A)

### **3. Metode Eliminasi Gauss-Jordan (Metode Langsung / Eksak)**

**Cara kerja secara deskriptif:**
1. Function menerima matriks `A` dan vektor `b`, lalu mengonversi keduanya ke tipe float.
2. Membuat matriks augmented $M = [A | b]$.
3. Untuk setiap kolom pivot $k = 0 \dots n-1$:
   - Normalisasi baris pivot agar elemen diagonal menjadi 1.
   - Eliminasi (mengenolkan) semua elemen di kolom $k$ baik di atas maupun di bawah diagonal.
4. Setelah proses selesai, matriks $A$ telah menjadi matriks identitas, dan kolom terakhir berisi solusi eksak $x$.

Metode ini menghasilkan solusi **eksak** (dalam batas floating-point) tanpa iterasi.

In [ ]:
def gauss_jordan(A, b):
    """
    Menyelesaikan sistem Ax = b menggunakan Eliminasi Gauss-Jordan.
    Mengembalikan vektor solusi x.
    """
    # Salin data agar tidak mengubah matriks asli
    A_work = np.array(A, dtype=float)
    b_work = np.array(b, dtype=float).flatten()
    n = len(b_work)

    # Buat matriks augmented [A | b]
    M = np.hstack((A_work, b_work.reshape(-1, 1)))

    # Proses eliminasi Gauss-Jordan
    for k in range(n):
        # Normalisasi baris pivot (membuat diagonal = 1)
        pivot = M[k, k]
        if abs(pivot) < 1e-12:
            raise ValueError(f"Pivot hampir nol pada baris {k}. Sistem mungkin singular.")
        M[k, :] = M[k, :] / pivot

        # Eliminasi elemen di atas dan di bawah pivot
        for i in range(n):
            if i != k:
                faktor = M[i, k]
                M[i, :] = M[i, :] - faktor * M[k, :]

    # Solusi berada di kolom terakhir
    solusi = M[:, n]
    return solusi

### **4. Metode Iterasi Gauss-Seidel (Metode Hampiran / Iteratif)**

**Cara kerja secara deskriptif:**
1. Tebakan awal $x^{(0)}$ (biasanya vektor nol).
2. Pada setiap iterasi $k$, hitung $x_i^{(k+1)}$ secara berurutan menggunakan nilai terbaru yang sudah tersedia:

$$
x_i^{(k+1)} = \frac{1}{a_{ii}} \left( b_i - \sum_{j=1}^{i-1} a_{ij} x_j^{(k+1)} - \sum_{j=i+1}^{n} a_{ij} x_j^{(k)} \right)
$$

3. Hitung galat relatif maksimum antar iterasi.
4. Berhenti jika galat < toleransi (`es`) atau mencapai `max_iter`.

Keunggulan: menggunakan nilai terbaru segera (lebih cepat konvergen dibanding Jacobi).

In [ ]:
def gauss_seidel(A, b, x0=None, es=1e-4, max_iter=1000, verbose=False):
    """
    Menyelesaikan sistem Ax = b menggunakan Iterasi Gauss-Seidel.
    
    Parameter:
        A, b      : matriks koefisien dan vektor konstanta
        x0        : tebakan awal (default = vektor nol)
        es        : toleransi galat relatif maksimum
        max_iter  : batas maksimum iterasi
        verbose   : jika True, cetak progress setiap iterasi
    
    Mengembalikan:
        x         : vektor solusi pendekatan
        iterasi   : jumlah iterasi yang dibutuhkan
        galat     : galat relatif maksimum terakhir
    """
    n = len(b)
    b_flat = np.array(b, dtype=float).flatten()
    A_work = np.array(A, dtype=float)

    # Inisialisasi tebakan awal
    if x0 is not None:
        x = np.array(x0, dtype=float).flatten()
    else:
        x = np.zeros(n)

    for iterasi in range(1, max_iter + 1):
        x_lama = x.copy()

        # Update setiap variabel secara berurutan (menggunakan nilai terbaru)
        for i in range(n):
            sigma = 0.0
            for j in range(n):
                if j != i:
                    sigma += A_work[i, j] * x[j]   # x[j] sudah nilai terbaru jika j < i
            x[i] = (b_flat[i] - sigma) / A_work[i, i]

        # Hitung galat relatif maksimum
        galat_maks = 0.0
        for i in range(n):
            if abs(x[i]) > 1e-12:  # hindari division by zero
                galat_relatif = abs((x[i] - x_lama[i]) / x[i])
                galat_maks = max(galat_maks, galat_relatif)

        if verbose and (iterasi <= 5 or iterasi % 10 == 0):
            print(f"Iterasi {iterasi:3d} | galat = {galat_maks:.6e} | x = {np.round(x, 4)}")

        # Cek konvergensi
        if galat_maks < es:
            if verbose:
                print(f"\nKonvergen pada iterasi ke-{iterasi} dengan galat = {galat_maks:.6e}")
            return x, iterasi, galat_maks

    print(f"Peringatan: Tidak konvergen dalam {max_iter} iterasi. Galat terakhir = {galat_maks:.6e}")
    return x, max_iter, galat_maks

### **5. Eksekusi Kedua Metode & Pengukuran Performa**

Kita akan:
- Menjalankan Gauss-Jordan (solusi eksak)
- Menjalankan Gauss-Seidel dengan toleransi $es = 10^{-4}$
- Mengukur waktu eksekusi masing-masing
- Membandingkan akurasi (galat relatif terhadap solusi Gauss-Jordan)

In [ ]:
# Metode Gauss-Jordan (Eksak)
print("=" * 60)
print("METODE ELIMINASI GAUSS-JORDAN")
print("=" * 60)

start_gj = time.perf_counter()
x_gj = gauss_jordan(A, b)
waktu_gj = time.perf_counter() - start_gj

print("\nSolusi eksak (Gauss-Jordan):")
for i, val in enumerate(x_gj, 1):
    print(f"  x{i} = {val:.10f} paket/detik")
print(f"\nWaktu eksekusi : {waktu_gj*1000:.4f} ms")

In [ ]:
# ------------------------------------------------------------
# 5.2  Metode Gauss-Seidel (Iteratif)
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("METODE ITERASI GAUSS-SEIDEL")
print("=" * 60)
print("Toleransi (es) = 1e-4 | Tebakan awal = [0, 0, 0, 0, 0, 0]")
print("-" * 60)

start_gs = time.perf_counter()
x_gs, n_iter, galat_akhir = gauss_seidel(A, b, es=1e-4, max_iter=1000, verbose=True)
waktu_gs = time.perf_counter() - start_gs

print("\nSolusi pendekatan (Gauss-Seidel):")
for i, val in enumerate(x_gs, 1):
    print(f"  x{i} = {val:.10f} paket/detik")
print(f"\nJumlah iterasi : {n_iter}")
print(f"Galat terakhir : {galat_akhir:.6e}")
print(f"Waktu eksekusi : {waktu_gs*1000:.4f} ms")

In [ ]:
# Tabel Perbandingan Performa
print("\n" + "=" * 70)
print("TABEL PERBANDINGAN PERFORMA")
print("=" * 70)

# Hitung galat relatif terhadap solusi Gauss-Jordan (dianggap sebagai "nilai sejati")
galat_relatif = np.abs((x_gs - x_gj) / x_gj) * 100  # dalam persen

print(f"{'Variabel':<10} {'Gauss-Jordan (eksak)':>22} {'Gauss-Seidel':>18} {'Galat Relatif (%)':>18}")
print("-" * 70)
for i in range(6):
    print(f"x{i+1:<9} {x_gj[i]:>22.10f} {x_gs[i]:>18.10f} {galat_relatif[i]:>17.8f}")

print("-" * 70)
print(f"{'Waktu (ms)':<10} {waktu_gj*1000:>22.4f} {waktu_gs*1000:>18.4f}")
print(f"{'Iterasi':<10} {'1 (langsung)':>22} {n_iter:>18}")
print(f"{'Tipe':<10} {'Eksak / Direct':>22} {'Hampiran / Iteratif':>18}")
print("=" * 70)

### **6. Analisis & Kesimpulan**

**Akurasi:**
- Gauss-Jordan menghasilkan solusi **eksak** (dalam batas presisi floating-point double).
- Gauss-Seidel dengan toleransi $10^{-4}$ sudah sangat dekat dengan solusi eksak (galat relatif jauh di bawah 0,01%).

**Kecepatan:**
- Untuk sistem kecil (6×6), Gauss-Jordan biasanya lebih cepat karena hanya sekali proses eliminasi.
- Gauss-Seidel membutuhkan beberapa puluh iterasi, tetapi setiap iterasi sangat ringan.

**Kapan memilih metode mana?**
| Situasi                              | Metode yang Disarankan     |
|--------------------------------------|----------------------------|
| Sistem kecil – sedang ($n \leq 100$) | Gauss-Jordan / LU          |
| Sistem sangat besar & sparse         | Gauss-Seidel / Iteratif    |
| Butuh solusi sangat akurat           | Gauss-Jordan               |
| Matriks diagonally dominant          | Gauss-Seidel sangat cocok  |

**Kesimpulan untuk kasus 6 router ini:**
Kedua metode berhasil menemukan solusi yang konsisten. Gauss-Jordan memberikan nilai “sejati”, sementara Gauss-Seidel memberikan pendekatan yang sangat baik dengan toleransi yang wajar. Karena matriks memenuhi syarat diagonally dominant, Gauss-Seidel konvergen dengan baik.